Bronze DDL

In [0]:
# create dynamic catalog and schems

spark.sql("CREATE CATALOG IF NOT EXISTS delivery_cata")
spark.sql("USE CATALOG delivery_cata")
spark.sql("CREATE SCHEMA IF NOT EXISTS quarantine")
spark.sql("CREATE SCHEMA IF NOT EXISTS delivery_ddl")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_ai")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_rpt")

DataFrame[]

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.areas_raw (
    area_id         INT,
    area_name       STRING,
    city            STRING,
    pincode         STRING,
    delivery_charge DOUBLE,
    _ingested_at    TIMESTAMP,
    _source_file    STRING
)USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.users_raw(
    user_id       INT,
    name          STRING,
    phone         STRING,
    email         STRING,
    address       STRING,
    created_at    DATE,
    _ingested_at  TIMESTAMP,
    _source_file  STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.drivers_raw(
    driver_id     INT,
    name          STRING,
    phone         STRING,
    vehicle_id    DOUBLE,
    area_id       DOUBLE,
    status        STRING,
    _ingested_at  TIMESTAMP,
    _source_file  STRING

    
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.vehicles_raw (
     vehicle_id    INT,
    vehicle_type  STRING,
    plate_number  STRING,
    capacity_kg   DOUBLE,
    status        STRING,
    _ingested_at  TIMESTAMP,
    _source_file  STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.orders_raw (
    order_id           INT,
    user_id            DOUBLE,
    pickup_location    STRING,
    delivery_location  STRING,
    order_amount       DOUBLE,
    order_status       STRING,
    created_at         DATE,
    _ingested_at       TIMESTAMP,
    _source_file       STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.deliveries_raw(
    delivery_id      INT,
    order_id         DOUBLE,
    driver_id        DOUBLE,
    vehicle_id       DOUBLE,
    pickup_time      TIMESTAMP,
    delivery_time    TIMESTAMP,
    delivery_status  STRING,
    distance_km      DOUBLE,
    _ingested_at     TIMESTAMP,
    _source_file     STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS delivery_cata.delivery_ddl.payments_raw(
    payment_id      INT,
    order_id        INT,
    user_id         DOUBLE,
    amount          DOUBLE,
    payment_method  STRING,
    payment_status  STRING,
    payment_time    TIMESTAMP,
    _ingested_at    TIMESTAMP,
    _source_file    STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'layer' = 'landing'
);

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS delivery_cata.quarantine.bad_rows (
        source_table  STRING,
        raw_data      STRING,
        error_reason  STRING,
        _ingested_at  TIMESTAMP,
        _batch_id     STRING
    ) USING DELTA
""")


DataFrame[]

Gold RPT ddl

In [0]:
spark.sql('USE CATALOG delivery_cata')
spark.sql('USE SCHEMA gold_rpt')


DataFrame[]

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.fact_delivery (
    -- Surrogate primary key
    delivery_sk BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Auto-incremented SK',

    -- Business keys (kept for traceability)
    delivery_id             INT,
    order_id                INT,

    -- Foreign keys to dimensions
    driver_sk               INT,
    customer_sk             INT,
    vehicle_sk              INT,
    area_sk                 INT,
    payment_sk              INT,
    date_sk                 INT,

    -- Degenerate dimensions (low-cardinality, no separate dim needed)
    order_status            STRING,
    delivery_status         STRING,
    pickup_location         STRING,
    delivery_location       STRING,

    -- Measures
    order_amount            DOUBLE,
    payment_amount          DOUBLE,
    distance_km             DOUBLE,
    delivery_time_mins      INT,
    delivery_charge         DOUBLE,

    -- Date/time stamps (for time-series analysis)
    pickup_time             TIMESTAMP,
    delivery_time           TIMESTAMP,
    order_date              DATE,
    payment_time            TIMESTAMP,

    -- Metadata
    _loaded_at              TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'layer'          = 'gold_rpt',
    'ai_queryable'   = 'true',
    'schema_version' = '1.0'
)
""")
print('fact_delivery created')

fact_delivery created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_payment (
    payment_sk          INT,
    payment_id          INT,
    payment_method      STRING,
    payment_status      STRING,
    payment_time        TIMESTAMP,
    _loaded_at          TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rpt', 'ai_queryable'='true')
""")
print('dim_payment created')

dim_payment created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_area (
    area_sk             INT,
    area_id             INT,
    area_name           STRING,
    city                STRING,
    pincode             STRING,
    delivery_charge     DOUBLE,
    _loaded_at          TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rpt', 'ai_queryable'='true')
""")
print('dim_area created')

dim_area created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_vehicle (
    vehicle_sk          INT,
    vehicle_id          INT,
    vehicle_type        STRING,
    plate_number        STRING,
    capacity_kg         DOUBLE,
    vehicle_status      STRING,
    _loaded_at          TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rpt', 'ai_queryable'='true')
""")
print('dim_vehicle created')

dim_vehicle created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_customer (
    customer_sk         INT,
    customer_id         INT,
    customer_name       STRING,
    customer_email      STRING,
    customer_phone      STRING,
    customer_address    STRING,
    customer_since      DATE,

    effective_start     TIMESTAMP,
    effective_end       TIMESTAMP,
    is_current          BOOLEAN,
    _loaded_at          TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rpt', 'ai_queryable'='true')
""")
print('dim_customer created')



dim_customer created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_driver (
    driver_sk           INT,
    driver_id           INT,
    driver_name         STRING,
    driver_phone        STRING,
    driver_status       STRING,
    
    effective_start     TIMESTAMP,
    effective_end       TIMESTAMP,
    is_current          BOOLEAN,
    _loaded_at          TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rpt', 'ai_queryable'='true')
""")
print('dim_driver created')

dim_driver created


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_rpt.dim_date (
    date_sk         INT,
    full_date       DATE,
    year            INT,
    quarter         INT,
    month           INT,
    month_name      STRING,
    week_of_year    INT,
    day_of_month    INT,
    day_of_week     INT,
    day_name        STRING,
    is_weekend      BOOLEAN,
    _loaded_at      TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer'='gold_rptr', 'ai_queryable'='true')
""")
print('dim_date created')

dim_date created


Gold AI ddl

In [0]:
spark.sql('USE CATALOG delivery_cata')
spark.sql('USE SCHEMA gold_ai')


DataFrame[]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS delivery_cata.gold_ai")

gold_tables = [
    "master_delivery_fact",
    "driver_performance_daily", 
    "area_daily_summary",
    "customer_order_summary",
    "payment_summary",
    "vehicle_utilization"
]

In [0]:
# delivery fact table 
# denormalized fact table for AI model queries

spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.master_delivery_fact (
    -- delivery identifiers
    delivery_id             INT,
    order_id                INT,

    -- driver info
    driver_id               INT,
    driver_name             STRING,
    driver_phone            STRING,    
    driver_status           STRING,

    -- vehicle info
    vehicle_id              INT,
    vehicle_type            STRING,
    vehicle_plate_number    STRING,
    vehicle_capacity_kg     DOUBLE,     

    -- area info
    area_id                 INT,
    area_name               STRING,
    city                    STRING,
    pincode                 STRING,     
    delivery_charge         DOUBLE,

    -- customer info
    customer_id             INT,
    customer_name           STRING,
    customer_email          STRING,
    customer_phone          STRING,    
    customer_address        STRING,

    -- order info
    order_amount            DOUBLE,     
    order_status            STRING,
    pickup_location         STRING,
    delivery_location       STRING,
    order_date              DATE,

    -- delivery info
    delivery_status         STRING,
    pickup_time             TIMESTAMP,
    delivery_time           TIMESTAMP,
    distance_km             DOUBLE,    

    -- payment info
    payment_id              INT,
    payment_method          STRING,
    payment_status          STRING,
    payment_amount          DOUBLE,     
    payment_time            TIMESTAMP,

    -- metadata
    _loaded_at              TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")

DataFrame[]

In [0]:
 #DRIVER PERFORMANCE DAILY
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.driver_performance_daily (
    driver_id                   INT,
    driver_name                 STRING,
    vehicle_type                STRING,
    delivery_date               DATE,
    total_orders_assigned       LONG,
    total_delivered             LONG,
    total_failed                LONG,
    total_cancelled             LONG,
    success_rate_pct            DOUBLE,
    avg_delivery_time_mins      DOUBLE,
    avg_distance_km             DOUBLE,
    total_distance_km           DOUBLE,
    areas_covered               INT,
    _loaded_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")
print("created")

created


In [0]:
#AREA DAILY SUMMARY
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.area_daily_summary (
    area_id                     INT,
    area_name                   STRING,
    city                        STRING,
    pincode                     STRING, 
    delivery_date               DATE,
    total_orders                LONG,
    total_delivered             LONG,
    total_failed                LONG,
    total_cancelled             LONG,
    total_revenue               DOUBLE,
    avg_delivery_time_mins      DOUBLE,
    avg_distance_km             DOUBLE,
    unique_drivers              LONG,
    unique_customers            LONG,
    _loaded_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")
print("created")

created


In [0]:
#CUSTOMER ORDER SUMMARY
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.customer_order_summary (
    customer_id                 INT,
    customer_name               STRING,
    customer_email              STRING,
    customer_phone              STRING,     
    customer_address            STRING,
    total_orders                LONG,
    total_delivered             LONG,
    total_cancelled             LONG,
    total_spent                 DOUBLE,
    avg_order_value             DOUBLE,
    preferred_payment_method    STRING,
    most_ordered_area           STRING,
    first_order_date            DATE,
    last_order_date             DATE,
    _loaded_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")
print("created")


created


In [0]:
#PAYMENT SUMMARY
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.payment_summary (
    payment_date                DATE,
    payment_method              STRING,
    area_name                   STRING,
    city                        STRING,
    pincode                     STRING,     
    total_transactions          LONG,
    total_revenue               DOUBLE,
    avg_transaction_value       DOUBLE,
    completed_payments          LONG,
    failed_payments             LONG,
    pending_payments            LONG,
    _loaded_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")
print("Created")

Created


In [0]:
#VEHICLE UTILIZATION
spark.sql("""
CREATE TABLE IF NOT EXISTS delivery_cata.gold_ai.vehicle_utilization (
    vehicle_id                  INT,
    vehicle_type                STRING,
    vehicle_plate_number        STRING,
    vehicle_capacity_kg         DOUBLE,
    driver_id                   INT,
    driver_name                 STRING,
    utilization_date            DATE,
    total_deliveries            LONG,
    total_distance_km           DOUBLE,
    avg_distance_per_trip_km    DOUBLE,
    total_delivered             LONG,
    total_failed                LONG,
    _loaded_at                  TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('layer' = 'gold_ai', 'ai_queryable' = 'true')
""")
print("created")

created
